# Stage-0 checks of the noisy two-headed PFN

## Description

**Question.** Is the two-headed PFN from `noisy_pfn` (notebook 06) good enough to build noise-aware
acquisition on, and is the noise regime identifiable early in a run? This is the Stage-0 gate of the
"learned acquisition policies" plan: closed-loop adaptation to noise can only pay if the regime can be
read off the state early, at about t/T = 0.2. With T in [20, 80] and a small initial design, that means
context sizes of about 8 to 16.

**Setup.** Fresh functions (data seed 333,000, disjoint from training and its validation).
1. **Calibration.** 256 functions per regime; contexts of size n in {1, 2, 5, 10, 20, 50, 100} (prefixes of
   one generating sequence); 200 test points. Per head: NLL, 50% / 90% central coverage of its own target
   (f for latent, y for noisy), PIT histograms. **Control:** the noisy head read as a predictive for f.
   That is what an observation-level PFN offers for f, so the latent head should beat it, increasingly
   with noise.
2. **Total variance.** For Gaussian / Student-t / skew noise, Var_noisy - Var_latent against the true noise
   variance at the test points. The ratio and, for heteroscedastic noise, the rank correlation.
3. **Martingale self-test.** 256 functions, n in {5, 20, 50}: draw 32 y ~ noisy head at a random x,
   re-encode, average the new latent means at 64 probe points, compare with the current latent mean.
   Report the bias next to its Monte Carlo error and to how far one observation moves the mean.
4. **Identifiability probes.** 2048 functions with prior-sampled properties, n in
   {2, 4, 8, 12, 16, 24, 32, 48}. Fourteen interpretable features from one encoding: the noise estimate
   sqrt(Var_noisy - Var_latent) over 256 probe points (mean, log, CV), latent and noisy spread, an SNR
   estimate, standardized residuals at the context points, noisy-head skewness and kurtosis, and the noise
   estimate at the best point. Linear probes are trained on even functions and scored on odd ones; the
   null control is the majority class or the training mean. Targets: regime (3 classes), noisy vs free,
   hetero vs homo, family, log10 SNR, d_eff.

**What to look for.** Coverage near nominal in every regime and n; the latent-vs-control NLL gap growing as
SNR falls; probe accuracy rising with n, and how early.

**GO / NO-GO** (pre-registered, per d, at the pilot):
- **Calibration** (required): 90% coverage of each head within [0.85, 0.95] in every regime at n in
  {5, 20, 50}. The latent head's NLL for f is below the control's, in the noisy regimes at n >= 5, with a
  paired bootstrap CI that excludes 0.
- **Identifiability** (the plan's Stage-0 gate), at n = 12:
  - balanced accuracy of the regime probe >= 0.70 (null 0.33);
  - noisy vs free >= 0.85;
  - R^2 of log10 SNR >= 0.5.
- **GO** if both hold. **INCONCLUSIVE** if calibration holds but identifiability reaches only the null +
  0.15 or more, and not the threshold. **NO-GO** if calibration fails, or the probes stay within 0.15 of
  the null at n = 12.
- Total variance and the martingale self-test are diagnostics, reported but not gated.

**Limits.**
- Linear probes on hand-made features are a lower bound on what the encoding contains.
- Probe contexts come from the prior's context generator, a stand-in for real BO trajectories; the BO
  side (anytime-acquisition) re-checks on its own trajectories.
- One checkpoint per d (pilot); in-prior only, so misspecification is not tested here.


In [ ]:
from pfn_repro.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/eval_noisy_pfn/<sweep_id>"  # set after `just snapshot runs/eval_noisy_pfn/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.